In [ ]:
#코랩에서 한글시각화설정
!pip install -q koreanize-matplotlib
import koreanize_matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 14.4 MB/s eta 0:00:00


In [ ]:
import tensorflow as tf
print(tf.__version__)

2.21.0


In [ ]:
import torch
print("gpu" if torch.cuda.is_available() else "cpu")
print('torch버전:',torch.__version__)

cpu
torch버전: 2.11.0+cpu


# DNN을 활용한 심장병 발병예측(과거에는 텐서플로로 했음. 이제는 파이토치로 함)
1. 데이터 불러오기
- 결측치 처리(이미 완료)
- 원-핫 인코딩(y는 이미 완료. X도 쉽게 하는 방법도 있음)
- 원핫은, 변수가 카테코리화면인 경우임. 표준정규화는 연속형
- csv백업(구글드라이브 특정 폴더에 저장할 것임. 그냥 세이브하면 닫으며 다 날라감.)
2. feature(X)와 타겟(y)을 분리
3. 훈련/테스트 데이터 분할함 (==> 여기까지는 파이토치도 동일함)
4. Dataset클래스 정의해야 함. 그리고 데이터로더도 생성해야 함. 즉, 텐서플로우는 바로 모델 들어감. 파이토치는 데이터 설계작업을 들어가야 함
5. 모델정의도 클래스화 시킴. 파이토치, 텐서 등을 모두다 딥러닝. 머신러닝 다 해보는 것
6. 손실하마수와 옵티마이저를 세팅(ex. 렐루, 아담, 등등)
7. 학습루프를 돌림(파이토치는 내가 설정해줘야 함. 텐서에서는 fit만 하면 됨. 쉬움. 단, 텐서 1버전에서는 for문을 내가 돌림 이렇게 파이토치를 해야함. 불편)
8. 학습끝나면 평가(테스트데이터로 함). 32개씩 가져와서 해야 하는데 배치싸이즈 32할거면 트레인, 테스트 모두다 해야 함
9. 예측

#1. 데이터불러오기

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn #Sequential(), Dropout(), Linear()
import torch.optim as optim

In [ ]:
# 심장병 데이터셋 파일 불러오기
# https://github.com/4aix/data/blob/master/heart.csv =>
# https://raw.githubusercontent.com/4aix/data/master/heart.csv
url = 'https://raw.githubusercontent.com/4aix/data/master/heart.csv'
df = pd.read_csv(url)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        303 non-null    int64  
 12  thal      303 non-null    int64  
 13  target    303 non-null    int64  
dtypes: float64(1), int64(13)
memory usage: 33.3 KB


## 데이터전처리
- 카테코리칼 변수는 원핫인코딩
- 연속형 수치 변수는 정규화(MinMaxScaler)..내 마음이다.
총 14개 변수 중 categorical 변수는 4개이다

In [ ]:
# feature변수의 원핫인코딩
from pandas.core.arrays import categorical
# ['cp', 'restecg', 'slope', 'thal' ] 카테고리컬한 변수 컬럼이름
categorical_cols = ['cp', 'restecg', 'slope', 'thal' ]
# 여러 열을 동시에 원핫인코딩
df = pd.get_dummies(df, columns=categorical_cols)
df.shape

(303, 24)

In [ ]:
# feature의 scale 조정
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
minmax_scaler = MinMaxScaler()
numeric_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']
# df = df.copy()
df[numeric_cols] = minmax_scaler.fit_transform(df[numeric_cols])
df.head()

,age,sex,trestbps,chol,fbs,thalach,exang,oldpeak,ca,target,...,restecg_0,restecg_1,restecg_2,slope_0,slope_1,slope_2,thal_0,thal_1,thal_2,thal_3
0,0.708333,1,0.481132,0.244292,1,0.603053,0,0.370968,0.0,1,...,True,False,False,True,False,False,False,True,False,False
1,0.166667,1,0.339623,0.283105,0,0.885496,0,0.564516,0.0,1,...,False,True,False,True,False,False,False,False,True,False
2,0.250000,0,0.339623,0.178082,0,0.770992,0,0.225806,0.0,1,...,True,False,False,False,False,True,False,False,True,False
3,0.562500,1,0.245283,0.251142,0,0.816794,0,0.129032,0.0,1,...,False,True,False,False,False,True,False,False,True,False
4,0.583333,0,0.245283,0.520548,0,0.702290,1,0.096774,0.0,1,...,False,True,False,False,False,True,False,False,True,False


In [ ]:
df.to_csv('/content/drive/MyDrive/(주)한국능률협회컨설팅_aix/파인튜닝/heart_preprocessed.csv',index=False)

# 2. feature와 target분리

In [ ]:
X = df.drop(columns=['target']).astype(np.float32)
y = df['target']
X.shape, y.shape

((303, 23), (303,))

In [ ]:
y.value_counts(normalize=True)

,proportion
target,
1,0.544554
0,0.455446


# 3. 훈련/테스트 데이터 분리

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X.values,  # 넘파이배열로 변환
                                                    y.to_numpy(),
                                                    test_size=0.2,
                                                    random_state=7,
                                                    stratify=y # 타겟비율 유지
                                                    )
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(242, 23) (242,) (61, 23) (61,)


In [ ]:
torch.tensor(y, dtype=torch.float32) # 1차원 텐서
#torch.tensor(y, dtype=torch.float32).view(-1, 1) # 2차원 텐서

tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 

# 4. Dataset 클래스, 데이터로더 생성
- 데이터로더 : batch_size만큼 데이터를 가져오는 객채

In [ ]:
from torch.utils.data import Dataset, DataLoader, TensorDataset
class HeartDataSet(Dataset):
    # 생성자 : X, y를 텐서 형태로 변환하여 저장
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1)
    # 데이터 셋의 전체 길이를 반환
    def __len__(self):
        return len(self.y)
    # 주어진 index에 해당하는 데이터(X, y) 쌍을 반환
    def __getitem__(self, index):
        return self.X[index], self.y[index]

In [ ]:
# 데이터 셋 객체
train_dataset = HeartDataSet(X_train, y_train)
test_dataset  = HeartDataSet(X_test, y_test)
# 데이터 로더 생성(32 batchsize씩 매 epoch마다 데이터 섞어서 로더)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [ ]:
len(X_train)/32

7.5625

In [ ]:
# train_loader : batch_size단위로 데이터를 반복해서 공급하는지
for feature, label in train_loader:
    print(len(feature), len(label))
print(len(train_loader), '개 batch')

32 32
32 32
32 32
32 32
32 32
32 32
32 32
18 18
8 개 batch


# 5. 모델 정의

In [ ]:
class DNN(nn.Module):
    def __init__(self, input_dim):
        super(DNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
            # nn.sigmoid() 제거 : loss function에서 sigmoid를 고려한 처리
        )
    def forward(self, x):
        return self.net(x)
model = DNN(input_dim=X.shape[1]) # feature수 23

In [ ]:
# feature수, 독립변수 수, 입력변수 수, 입력뉴런 수
X.shape[1], X_train.shape[1], X_test.shape[1]

(23, 23, 23)

# 6. 손실함수 객체 및 옵티마이저 객체 설정

In [ ]:
# 손실함수 객체 (이진분류를 위한 가장 안전하고 권장됨)
criterion = nn.BCEWithLogitsLoss()
# 옵티마이저
optimizer = optim.SGD(model.parameters(),
                      lr=0.01,
                      momentum=0.9)
# 학습률(lr)이 0~50까지 서서히 줄어들게
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer,
                                                 T_max=50)

# 7. 학습 루프

In [ ]:
epochs = 50
for epoch in range(epochs):
    # 모델을 학습 모드 전환
    model.train()
    # 한 epoch마다 출력될 손실값 초기화
    running_loss = 0
    # batch단위로 학습데이터 불러와 학습 수행
    for X_batch, y_batch in train_loader:
        # 이전 단계의 기울기 초기화
        optimizer.zero_grad()
        # 모델 예측값
        outputs = model(X_batch)
        # 손실함수 객체로 loss 계산
        loss = criterion(outputs, y_batch)
        # 역전파 수행하여 기울 계산하고, 파라미터 업데이트
        loss.backward()
        optimizer.step()
        running_loss += loss.item() # loss텐서안의 값만
    #  스케줄러를 통한 학습률 조정
    scheduler.step()
    # 한 epoch의 평균 loss 출력
    avg_loss = running_loss / len(train_loader)
    # 5 epoch마다 loss 출력
    if (epoch+1) % 5 == 0:
        print(f'{epoch+1}번째 : loss {avg_loss:.4f}')

5번째 : loss 0.2524
10번째 : loss 0.2604
15번째 : loss 0.2305
20번째 : loss 0.2244
25번째 : loss 0.2128
30번째 : loss 0.2038
35번째 : loss 0.2322
40번째 : loss 0.2034
45번째 : loss 0.2156
50번째 : loss 0.2027


# 8. 모델 평가

In [ ]:
# 평가 모드 전환
model.eval()
# 예측값과 실제값
y_pred = []
y_true = []
# 평가시에는 기술계산을 비활성화(메모리 사용량 및 속도가 향상)
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        # 모델 예측값 계산(sigmoid 적용 전)
        outputs = model(X_batch)
        # sigmoid를 적용한 예측값(확률)
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()
        y_pred.extend(preds)
        y_true.extend(y_batch.numpy().tolist())

In [ ]:
from sklearn.metrics import ( accuracy_score, precision_score, f1_score,
                             recall_score, confusion_matrix)
print('평가 결과')
print('Accuracy : ', accuracy_score(y_true, y_pred))
print('recall : ', recall_score(y_true, y_pred))
print('precision : ', precision_score(y_true, y_pred))
print('f1_score : ', f1_score(y_true, y_pred))

평가 결과
Accuracy :  0.7868852459016393
recall :  0.7878787878787878
precision :  0.8125
f1_score :  0.8


In [ ]:
confusion_matrix(y_true, y_pred)

array([[22,  6],
       [ 7, 26]])

# 9. 모델 사용(예측)

In [ ]:
sample = X_test[0]
x = torch.tensor(sample, dtype=torch.float32)
model.eval()
with torch.no_grad():
    logit = model(x)
    prob = torch.sigmoid(logit).item()
print("발병 예측" if prob >0.5 else "정상 예측")

발병 예측


In [ ]:
y_test[0]

np.int64(1)